In [1]:
from dataclasses import dataclass, field
from collections import defaultdict
import math
from typing import Dict, Set, List, Optional
from abc import ABC, abstractmethod
import random

In [10]:
class Node(ABC):
  @abstractmethod
  def find_children(self)->set['Node']:
    "Return all possible child nodes"

  @abstractmethod
  def get_random_child(self)->Optional['Node']:
    'Return a random child node'

  @abstractmethod
  def is_terminal(self)->bool:
    "Check if the state is a terminal node"

  @abstractmethod
  def reward(self)->float:
    "return the reward"

  @abstractmethod
  def __hash__(self)->int:
    "Define a hash func for the node"

  @abstractmethod
  def __eq__(self, other: object)->bool:
    "Compare to objects"


In [19]:
@dataclass
class MCTS:
  """Monte Carlo Tree Search implementation."""
  exploration_weight: float = 1.0
  total_rewards: Dict[Node, float] = field(default_factory=lambda: defaultdict(float))
  visit_counts: Dict[Node, int] = field(default_factory=lambda: defaultdict(int))
  children: Dict[Node, Set[Node]] = field(default_factory=lambda: defaultdict(set))

  def select_best_move(self, node: Node) -> Node:
    if node.is_terminal():
      raise ValueError(f'cannot select best node from terminal node {node}')
    if node not in self.children:
      return node.get_random_child()

  def simulate(self, node: Node) -> float:
    path = self._traverse_tree(node)
    leaf = path[-1]
    self._expand_node(leaf)
    reward = self._simulate_random_playout(leaf)
    self._backpropagate(path, reward)

  def _traverse_tree(self, node: Node) -> List[Node]:
    path = []
    while True:
      path.append(node)
      if node not in self.children or not self.children[node]:
        return path
      unexplored = self.children[node] - set(self.children.keys())
      if unexplored:
        n = unexplored.pop()
        path.append(n)
        return path
      node = self._select_uct(node)

  def _expand_node(self, node: Node):
    if node in self.children:
      return
    self.children[node] = node.find_children()

  def _simulate_random_playout(self, node: Node) -> float:
    invert_reward = True
    while True:
      if node.is_terminal():
        reward = node.reward()
        return 1 - reward if invert_reward else reward
      node = node.get_random_child()
      invert_reward = not invert_reward

  def _backpropagate(self, path: List[Node], reward: float):
    for node in reversed(path):
      self.total_rewards[node] += reward
      self.visit_counts[node] += 1
      reward = 1 - reward

  def _select_uct(self,  node: Node)->Node:
    assert(all(n in self.children for n in self.children[node]))
    log_n_parent = math.log(self.visit_counts[node])

    def uct(n: Node)->float:
      return self.total_rewards[n] / self.visit_counts[n] + self.exploration_weight * math.sqrt(log_n_parent / self.visit_counts[n])
    return max(self.children[node], key=uct)

  def _calculate_node_score(self, node: Node)->float:
    if self.visit_counts[node]==0:
      return float('inf')
    return self.total_rewards[node] / self.visit_counts[node]

In [20]:
class TicTacToeNode(Node):
    def __init__(self, state: str, player: str):
        self.state = state
        self.player = player

    def find_children(self) -> Set["TicTacToeNode"]:
        if self.is_terminal():
            return set()
        return {
            TicTacToeNode(
                self.state[:i] + self.player + self.state[i + 1 :],
                "O" if self.player == "X" else "X",
            )
            for i, value in enumerate(self.state)
            if value == " "
        }

    def get_random_child(self) -> Optional["TicTacToeNode"]:
        if self.is_terminal():
            return None
        empty_spots = [i for i, value in enumerate(self.state) if value == " "]
        index = random.choice(empty_spots)
        return TicTacToeNode(
            self.state[:index] + self.player + self.state[index + 1 :],
            "O" if self.player == "X" else "X",
        )

    def is_terminal(self) -> bool:
        return self.winner() is not None or " " not in self.state

    def reward(self) -> float:
        winner = self.winner()
        if winner is None:
            return 0.5  # Draw
        return 1.0 if winner == self.player else 0.0

    def __hash__(self) -> int:
        return hash(self.state)

    def __eq__(self, other: object) -> bool:
        return isinstance(other, TicTacToeNode) and self.state == other.state

    def winner(self) -> Optional[str]:
        lines = [
            (0, 1, 2),
            (3, 4, 5),
            (6, 7, 8),  # Rows
            (0, 3, 6),
            (1, 4, 7),
            (2, 5, 8),  # Columns
            (0, 4, 8),
            (2, 4, 6),  # Diagonals
        ]
        for line in lines:
            if self.state[line[0]] == self.state[line[1]] == self.state[line[2]] != " ":
                return self.state[line[0]]
        return None

In [21]:
def play_game():
    state = " " * 9
    mcts = MCTS()
    board = TicTacToeNode(state, "X")

    print("Initial board:")
    print_board(board.state)

    while True:
        # Human player's turn (O)
        human_move = int(input("Enter your move (0-8): "))
        board = TicTacToeNode(
            board.state[:human_move] + "O" + board.state[human_move + 1 :], "X"
        )
        print("\nBoard after your move:")
        print_board(board.state)

        if board.is_terminal():
            break

        # AI player's turn (X)
        for _ in range(1000):  # Number of MCTS iterations
            mcts.simulate(board)
        board = mcts.select_best_move(board)
        print("\nBoard after AI move:")
        print_board(board.state)

        if board.is_terminal():
            break

    winner = board.winner()
    if winner:
        print(f"\nPlayer {winner} wins!")
    else:
        print("\nIt's a draw!")


def print_board(state: str):
    for i in range(0, 9, 3):
        print(" ".join(state[i : i + 3]))

In [ ]:
class MCTSFixed(MCTS):
    def select_best_move(self, node: Node) -> Node:
        if node.is_terminal():
            raise ValueError(f'cannot select best node from terminal node {node}')

        self._expand_node(node)

        if not self.children[node]:
            return node.get_random_child()

        best_child = None
        max_visits = -1

        for child in self.children[node]:
            if self.visit_counts[child] > max_visits:
                max_visits = self.visit_counts[child]
                best_child = child
            elif self.visit_counts[child] == max_visits and best_child is not None:
                pass

        if best_child is None and self.children[node]:
            return random.choice(list(self.children[node]))
        elif best_child is None:
            return node.get_random_child()

        return best_child

def play_game_fixed():
    state = " " * 9
    mcts = MCTSFixed()
    board = TicTacToeNode(state, "X")

    print("Initial board:")
    print_board(board.state)

    while True:
        human_move = int(input("Enter your move (0-8): "))
        if not (0 <= human_move < 9) or board.state[human_move] != ' ':
            print("Invalid move. Please choose an empty spot between 0 and 8.")
            continue
        board = TicTacToeNode(
            board.state[:human_move] + "O" + board.state[human_move + 1 :], "X"
        )
        print("\nBoard after your move:")
        print_board(board.state)

        if board.is_terminal():
            break

        print("\nAI is thinking...")
        for _ in range(1000):  # Number of MCTS iterations
            mcts.simulate(board)

        ai_move_node = mcts.select_best_move(board)
        if ai_move_node is None:
            print("AI could not determine a move. Game might be in an unexpected state.")
            break
        board = ai_move_node

        print("\nBoard after AI move:")
        print_board(board.state)

        if board.is_terminal():
            break

    winner = board.winner()
    if winner:
        print(f"\nPlayer {winner} wins!")
    else:
        print("\nIt's a draw!")

play_game_fixed()

Initial board:
     
     
     
